<a href="https://colab.research.google.com/github/GuruPatel45/codealpha_task2/blob/main/codealpha_task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""CodeAlpha Task 4 - Real-Time Object Detection using YOLOv8 & Gradio (Premium UI)."""

!pip install -q gradio ultralytics opencv-python-headless

import gradio as gr
from ultralytics import YOLO
import cv2
import numpy as np
from collections import Counter

print("Loading YOLOv8 model... (first run downloads weights)")
detector = YOLO("yolov8n.pt")
print("Model ready!")


def summarize_detections(results) -> str:
    """Build a small HTML summary of what was found in the frame."""
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return '<div class="empty-msg">No objects detected. Try another image.</div>'

    names = results[0].names
    labels = [names[int(c)] for c in boxes.cls]
    confs = boxes.conf.tolist()
    counts = Counter(labels)
    avg_conf = round((sum(confs) / len(confs)) * 100)

    chips = "".join(
        f'<span class="obj-chip">{label} <b>×{count}</b></span>'
        for label, count in sorted(counts.items(), key=lambda x: -x[1])
    )
    total = len(labels)
    return (
        f'<div class="summary-head">'
        f'<span>✓ {total} object(s) detected</span>'
        f'<span class="avg-conf">avg. confidence {avg_conf}%</span></div>'
        f'<div class="chip-row">{chips}</div>'
    )


def run_detection(image, confidence):
    if image is None:
        return None, '<div class="empty-msg">Upload or capture an image to begin.</div>'

    results = detector(image, conf=confidence, verbose=False)
    annotated_bgr = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    return annotated_rgb, summarize_detections(results)


# ---------------------------------------------------------------------------
# Premium CSS — dark charcoal + gold/glass accent theme (compact, single-page)
# ---------------------------------------------------------------------------
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=Inter:wght@400;500;600&display=swap');

html, body { height: 100%; }

body, .gradio-container {
    background: radial-gradient(circle at 20% 0%, #1a1a1f 0%, #0c0c10 55%, #08080a 100%) !important;
    font-family: 'Inter', sans-serif !important;
}
.gradio-container {
    max-width: 1500px !important;
    width: 100% !important;
    margin: 10px auto !important;
    padding: 0 16px !important;
}

/* ---------- Panels ---------- */
.panel {
    border-radius: 14px !important;
    border: 1px solid rgba(212,175,55,0.18) !important;
    background: rgba(255,255,255,0.02) !important;
}
label span { color: #d4d4d8 !important; font-size: 12px !important; font-weight: 500 !important; }
.panel span, .panel p { color: #a1a1aa !important; }

/* Tighten default Gradio block spacing */
.gap { gap: 8px !important; }
.form { gap: 6px !important; }

/* Slider accent */
input[type="range"] { accent-color: #d4af37 !important; }

/* Buttons */
button.primary {
    background: linear-gradient(135deg, #d4af37, #b8912b) !important;
    color: #14140f !important; font-weight: 700 !important; border: none !important;
    box-shadow: 0 6px 16px rgba(212,175,55,0.25) !important;
}
button.primary:hover { filter: brightness(1.08); }
button:not(.primary) {
    background: rgba(255,255,255,0.04) !important; color: #d4d4d8 !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
}
button { min-height: 34px !important; padding: 6px 14px !important; }

/* ---------- Summary panel ---------- */
#summary-box {
    background: rgba(212,175,55,0.05); border: 1px solid rgba(212,175,55,0.22);
    border-radius: 12px; padding: 10px 14px; margin-top: 6px; min-height: 36px;
}
.summary-head {
    display: flex; justify-content: space-between; align-items: center;
    font-weight: 700; color: #f5d488; font-size: 12.5px; margin-bottom: 6px;
}
.avg-conf { color: #a1a1aa; font-size: 10.5px; font-weight: 500; }
.chip-row { display: flex; flex-wrap: wrap; gap: 6px; }
.obj-chip {
    background: rgba(212,175,55,0.1); color: #f5d488; border: 1px solid rgba(212,175,55,0.35);
    font-size: 11px; font-weight: 500; padding: 3px 10px; border-radius: 999px;
}
.obj-chip b { color: #fff; }
.empty-msg { color: #71717a; font-size: 12px; font-style: italic; }

#footer-note { text-align: center; font-size: 10px; color: #52525b; margin-top: 8px; letter-spacing: .3px; }
footer { visibility: hidden; }
"""

theme = gr.themes.Base(
    primary_hue="yellow",
    neutral_hue="zinc",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    body_background_fill="#0c0c10",
    block_background_fill="transparent",
    block_border_width="0px",
    button_primary_background_fill="linear-gradient(135deg, #d4af37, #b8912b)",
    button_primary_text_color="#14140f",
)

with gr.Blocks(css=custom_css, theme=theme, title="VisionScan Elite — Object Detector") as demo:

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(
                sources=["upload"], type="numpy",
                label="Source Image", elem_classes="panel", height=280,
            )
        with gr.Column(scale=1):
            output_img = gr.Image(
                type="numpy", label="Detected Objects",
                interactive=False, elem_classes="panel", height=280,
            )

    conf_slider = gr.Slider(
        minimum=0.1, maximum=0.9, value=0.4, step=0.05,
        label="Detection sensitivity",
    )
    with gr.Row():
        scan_btn = gr.Button("Run Detection", variant="primary")
        reset_btn = gr.ClearButton(value="Reset")

    result_summary = gr.HTML(
        '<div class="empty-msg">Results will appear here.</div>', elem_id="summary-box"
    )

    reset_btn.add([input_img, output_img])
    scan_btn.click(fn=run_detection, inputs=[input_img, conf_slider], outputs=[output_img, result_summary])

    gr.HTML('<div id="footer-note">VISIONSCAN ELITE · YOLOv8 · GRADIO · OPENCV</div>')

if __name__ == "__main__":
    demo.launch(share=True)


Loading YOLOv8 model... (first run downloads weights)
Model ready!


/tmp/ipykernel_570/880375831.py:135: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=theme, title="VisionScan Elite — Object Detector") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b63bc51737103e7db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
